In [58]:
import json
import time
import pandas as pd
from gigachat import GigaChat
from google.colab import userdata, drive, files
from tqdm import tqdm
from tenacity import retry, stop_after_attempt, wait_exponential

In [59]:
drive.mount('/content/drive')
csv_path = '/content/drive/MyDrive/data-zav2.csv'
try:
    df = pd.read_csv(csv_path)
    if df.empty:
        raise ValueError("CSV файл пуст")
except FileNotFoundError:
    raise FileNotFoundError(f"Файл {csv_path} не найден")
except pd.errors.EmptyDataError:
    raise ValueError("CSV файл не содержит данных")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [60]:
if 'Review' not in df.columns:
    raise KeyError("Колонка 'Review' отсутствует в CSV")
reviews = df['Review'].dropna().astype(str)
if len(reviews) == 0:
    raise ValueError("Нет ни одного валидного отзыва после очистки")
results = []

In [61]:
key = userdata.get('sber_api_key')
if not key:
    raise ValueError("API ключ не найден в userdata")

In [67]:
#verify_ssl_certs=False из-за проблем с сертификатами GigaChat
giga = GigaChat(
    credentials=key,
    verify_ssl_certs=False,
    scope="GIGACHAT_API_PERS",
    model="GigaChat"
)

In [72]:
@retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1, min=2, max=10))
def analyze_review(review):
  prompt = (
  f"""Прочитай и  проанализируй текст отзыва на тональность автора и тему. """
  f"""Текст отзыва - {review}. """
  f"""Формат анализа 1. Тональность текста (позитивный, отрицательный, нейтральный) """
  f"""2. Тема отзыва. """
  f"""Дай ответ строго в формате JSON. Например: {{"Тональность" : "Позитивный", "Тема" : "Качество"}}"""
  )
  print(prompt)
  try:
    response = giga.chat(prompt)
    if not response:
      return 'api_error', 'api_error'
    answer = response.choices[0].message.content
    start = answer.find('{')
    end = answer.rfind('}') + 1
    if start == -1 or end == 0:
      return 'parse_error', 'parse_error'
    answer_2_0 = json.loads(answer[start:end])
    return answer_2_0.get('Тональность', 'unknown'), answer_2_0.get('Тема', 'other')
  except json.JSONDecodeError:
    return 'json_error', 'json_error'
  except Exception as e:
    print(f"Ошибка для отзыва '{review[:50]}': {e}")
    return 'exception', 'exception'

In [64]:
for i, review in enumerate(tqdm(reviews, desc="Анализ отзывов")):
  try:
    sentiment, topic = analyze_review(review)
    results.append({
        'Review': review,
        'sentiment': sentiment,
        'topic': topic
        })
  except Exception as e:
    print(f"\nОшибка для отзыва #{i}: {str(e)[:100]}")
    results.append({
        'Review': review,
        'sentiment': 'error',
        'topic': str(e)[:50]
        })
  time.sleep(0.5)

Анализ отзывов: 100%|██████████| 583/583 [09:12<00:00,  1.06it/s]


In [65]:
results_df = pd.DataFrame(results)
results_df.to_csv('results.csv', index=False)
results_df.to_json('results.json', orient='records', indent=2)
files.download('results.csv')
files.download('results.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>